# Language Models are Unsupervised Multitask Learners (GPT-2)

Replication of Radford, Wu, Child, Luan, Amodei and Sutskever (2019), *Language Models are
Unsupervised Multitask Learners* (the GPT-2 technical report), building on the
decoder-only Transformer of *Improving Language Understanding by Generative Pre-Training*
(Radford et al., 2018).

We implement a small GPT from scratch: a decoder-only Transformer with token and positional
embeddings, masked (causal) multi-head self-attention, and a language-model head trained to
predict the next character. Trained at character level on a small text corpus, it reproduces
the core behavior of an autoregressive language model: the training loss falls steadily and
the model generates fresh text with English-like words, spacing, and structure.

In [1]:
import math, urllib.request, torch, torch.nn as nn, torch.nn.functional as F
torch.manual_seed(0)
DEVICE = "cpu"

In [2]:
# Character-level corpus (Tiny Shakespeare).
url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
text = urllib.request.urlopen(url, timeout=30).read().decode("utf-8")
chars = sorted(set(text)); V = len(chars)
stoi = {c: i for i, c in enumerate(chars)}; itos = {i: c for c, i in stoi.items()}
data = torch.tensor([stoi[c] for c in text], dtype=torch.long)
n = int(0.9 * len(data)); train_data, val_data = data[:n], data[n:]
print("corpus chars:", len(text), "| vocab:", V)

corpus chars: 1115394 | vocab: 65


In [3]:
BLOCK, BATCH = 64, 32
def get_batch(split):
    d = train_data if split == "train" else val_data
    ix = torch.randint(len(d) - BLOCK - 1, (BATCH,))
    x = torch.stack([d[i:i+BLOCK] for i in ix])
    y = torch.stack([d[i+1:i+BLOCK+1] for i in ix])
    return x, y
print("input/target shapes:", [t.shape for t in get_batch("train")])

input/target shapes: [torch.Size([32, 64]), torch.Size([32, 64])]


In [4]:
N_EMB, N_HEAD, N_LAYER = 128, 4, 4
class Head(nn.Module):
    def __init__(self, hs):
        super().__init__()
        self.k = nn.Linear(N_EMB, hs, bias=False); self.q = nn.Linear(N_EMB, hs, bias=False)
        self.v = nn.Linear(N_EMB, hs, bias=False)
        self.register_buffer("tril", torch.tril(torch.ones(BLOCK, BLOCK)))
    def forward(self, x):
        B, T, C = x.shape
        k, q, v = self.k(x), self.q(x), self.v(x)
        att = q @ k.transpose(-2, -1) * k.size(-1) ** -0.5
        att = att.masked_fill(self.tril[:T, :T] == 0, float("-inf"))   # causal mask
        return torch.softmax(att, -1) @ v

class MultiHead(nn.Module):
    def __init__(self):
        super().__init__()
        self.heads = nn.ModuleList([Head(N_EMB // N_HEAD) for _ in range(N_HEAD)])
        self.proj = nn.Linear(N_EMB, N_EMB)
    def forward(self, x): return self.proj(torch.cat([h(x) for h in self.heads], -1))

class Block(nn.Module):
    def __init__(self):
        super().__init__()
        self.sa = MultiHead(); self.ff = nn.Sequential(nn.Linear(N_EMB, 4*N_EMB), nn.GELU(), nn.Linear(4*N_EMB, N_EMB))
        self.n1 = nn.LayerNorm(N_EMB); self.n2 = nn.LayerNorm(N_EMB)
    def forward(self, x):
        x = x + self.sa(self.n1(x)); return x + self.ff(self.n2(x))

In [5]:
class GPT(nn.Module):
    def __init__(self):
        super().__init__()
        self.tok = nn.Embedding(V, N_EMB); self.pos = nn.Embedding(BLOCK, N_EMB)
        self.blocks = nn.Sequential(*[Block() for _ in range(N_LAYER)])
        self.ln = nn.LayerNorm(N_EMB); self.head = nn.Linear(N_EMB, V)
    def forward(self, idx):
        B, T = idx.shape
        x = self.tok(idx) + self.pos(torch.arange(T))
        return self.head(self.ln(self.blocks(x)))
    @torch.no_grad()
    def generate(self, idx, n):
        for _ in range(n):
            logits = self(idx[:, -BLOCK:])[:, -1, :]
            idx = torch.cat([idx, torch.multinomial(torch.softmax(logits, -1), 1)], 1)
        return idx

model = GPT()
print("parameters:", sum(p.numel() for p in model.parameters()))

parameters: 816705


In [6]:
opt = torch.optim.AdamW(model.parameters(), lr=3e-3)
for step in range(4000):
    x, y = get_batch("train")
    logits = model(x)
    loss = F.cross_entropy(logits.reshape(-1, V), y.reshape(-1))
    opt.zero_grad(); loss.backward(); opt.step()
    if (step+1) % 500 == 0:
        model.eval()
        with torch.no_grad():
            vx, vy = get_batch("val")
            vl = F.cross_entropy(model(vx).reshape(-1, V), vy.reshape(-1))
        model.train()
        print(f"step {step+1}: train loss {loss.item():.3f}  val loss {vl.item():.3f}")

step 500: train loss 1.707  val loss 1.823


step 1000: train loss 1.560  val loss 1.624


step 1500: train loss 1.452  val loss 1.641


step 2000: train loss 1.446  val loss 1.647


step 2500: train loss 1.336  val loss 1.616


step 3000: train loss 1.414  val loss 1.580


step 3500: train loss 1.434  val loss 1.639


step 4000: train loss 1.305  val loss 1.521


In [7]:
# Generate text from the trained model, starting from a newline.
model.eval()
start = torch.tensor([[stoi["\n"]]])
out = model.generate(start, 400)[0].tolist()
print("".join(itos[i] for i in out))



ISABELLA:
Takes me feart, and See her, thence and he
to point dread; Besides, by joy, then it standly
In at some wasted in this you supportion;
In her good sir.
I am hence to quoquench overther to have forfeits look,
Saucations put with the air deservorcess that obsoluties
After'd I aburate to ding in lie that his brother's week,
And speak their country's king these there.

LEONTES:
Achield my ho
